# Build the ancestry (HM3-restricted) panel (Google Batch / dsub)

One whole-cohort panel, every AoU sample with ACAF coverage -- no premade-label/`BASE_GROUP` pre-filter. Round 1's own broad gate (`03_round2_1000g_filter.ipynb`) does the real classification now, so pre-restricting here added nothing.

QC'd ACAF restricted to `01_build_1000g_reference.ipynb`'s HM3 SNP list, no thinning -- `05`'s GRM panel is QC'd + randomly thinned instead, unrelated to HM3; intersecting *that* against 1000G's HM3 set left only ~10K overlapping variants.

**ID+REF+ALT harmonized before extraction**, not just an ID-string `--extract` -- ACAF's own REF/ALT orientation can disagree with 1000G's at a site, and a bare ID match would silently drop that variant even though it's the same SNP.

Dsub/Batch only, same mechanism as `05` (`--image gcr.io/google.com/cloudsdktool/cloud-sdk:slim` + `gsutil -u "$PROJECT_ID"` for this Requester Pays bucket).

## Prerequisites

`dsub` installed, `gcloud` authenticated, `01_build_1000g_reference.ipynb` already run.

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Same project/bucket values as `05`. `KG_OUT_PREFIX`'s `1kg_all_qc.acount` (ID/REF/ALT for every HM3 QC'd variant) is what harmonization runs against.

In [ ]:
import os

# same values 05_genome_wide_qc_thinning_batch_submit.ipynb already confirmed working -- reused, not re-derived
PROJECT_ID = "wb-swift-sprout-7231"
REGION = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
WORKSPACE_BUCKET_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning (filename/log tags) throughout
# this notebook. Fixed literal, matches every other notebook in this pipeline.
PROJECT_DIR = "covariance_v9"

ACAF_BUCKET_GS = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
ANCESTRY_BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/{PROJECT_DIR}/01_ancestry_filtering"

KG_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # 01_build_1000g_reference.ipynb's output, CDR-independent
KG_OUT_PREFIX = f"{KG_DIR}/1kg_all_qc"
assert os.path.isfile(f"{KG_OUT_PREFIX}.acount"), f"missing {KG_OUT_PREFIX}.acount -- run 01_build_1000g_reference.ipynb first"

# single whole-cohort panel -- no BASE_GROUP, no premade-label restriction
BUCKET_DIR_GS = f"{ANCESTRY_BUCKET_DIR_GS}/ancestry_panel"
PLINK_BIN_GS = f"{BUCKET_DIR_GS}/bin/plink2"
KG_HARMONIZE_GS = f"{BUCKET_DIR_GS}/1kg_id_ref_alt.sorted"

# per-task machine -- same sizing 05 landed on after chr1-4's real OOM/disk
# failure on a smaller machine (n1-standard-8, --disk-size 100)
MACHINE_VCPUS = 16
MEMORY_MB = 55000

print(ACAF_BUCKET_GS)
print(BUCKET_DIR_GS)

## Build and stage the 1000G ID+REF+ALT harmonization table

Sorted `ID REF ALT` from `1kg_all_qc.acount` -> staged to the bucket (not Requester Pays) for `--input`. Each per-chromosome task below does its own `comm -12` against this after relabeling ACAF's IDs -- true harmonization, not a bare ID match.

In [ ]:
%%bash -s "$KG_OUT_PREFIX" "$KG_HARMONIZE_GS"
set -e
KG_OUT_PREFIX=$1
KG_HARMONIZE_GS=$2

LOCAL_TABLE="/tmp/1kg_id_ref_alt.sorted"
awk 'NR>1 {print $2, $3, $4}' "${KG_OUT_PREFIX}.acount" | LC_ALL=C sort > "$LOCAL_TABLE"
wc -l "$LOCAL_TABLE"

gcloud storage cp "$LOCAL_TABLE" "$KG_HARMONIZE_GS"
gcloud storage ls -l "$KG_HARMONIZE_GS"

## Stage the plink2 binary (one-time)

No `wget`/`curl` on the default image -- stage once, localize via `--input`.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

"$BIN_DIR/plink2" --version

In [ ]:
%%bash -s "$PLINK_BIN_GS"
set -e
PLINK_BIN_GS=$1

local_plink="$HOME/bin/plink2"
if [ ! -x "$local_plink" ]; then
  echo "no local plink2 at $local_plink -- run the cell above first" >&2
  exit 1
fi

gcloud storage cp "$local_plink" "$PLINK_BIN_GS"
gcloud storage ls -l "$PLINK_BIN_GS"

## Stage 1 -- single-chromosome validation

chr22. Each task: relabel ACAF's own IDs to `chrom:pos:ref:alt` first, dedupe/biallelic-filter, build a sorted `ID REF ALT` table from *that*, `comm -12` against the staged 1000G table -- only then `--extract` the real agreeing set, before the rest of QC. Compare the output variant count against `1kg_all_qc.bim`'s per-chromosome count.

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$ACAF_BUCKET_GS" "$PLINK_BIN_GS" "$KG_HARMONIZE_GS" "$BUCKET_DIR_GS" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
ACAF_BUCKET_GS=$8
PLINK_BIN_GS=$9
KG_HARMONIZE_GS=${10}
BUCKET_DIR_GS=${11}
MACHINE_VCPUS=${12}
MEMORY_MB=${13}
ANCESTRY_BUCKET_DIR_GS=${14}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"
CHR=22
OUT_NAME="chr${CHR}_hm3_${CDR_VERSION}"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "hm3-panel-chr22-validation" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --input KG_HARMONIZE="$KG_HARMONIZE_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --env CHR_PGEN_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pgen" \
  --env CHR_PVAR_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pvar" \
  --env CHR_PSAM_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.psam" \
  --env OUT_NAME="$OUT_NAME" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --make-pgen \
      --out chr_relabeled

    grep -v "^##" chr_relabeled.pvar | awk "NR>1 {print \$3, \$4, \$5}" | LC_ALL=C sort > chr_id_ref_alt.sorted
    LC_ALL=C comm -12 chr_id_ref_alt.sorted "$KG_HARMONIZE" | awk "{print \$1}" > agreeing_snps.ids
    echo "Agreeing with 1000G (ID+REF+ALT): $(wc -l < agreeing_snps.ids)"

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --extract agreeing_snps.ids \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  > /tmp/hm3_validation_job_id.txt

cat /tmp/hm3_validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
JOB_ID=$(tail -1 /tmp/hm3_validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

echo
echo "--- output files ---"
gcloud storage ls -l "${BUCKET_DIR_GS}/" 2>/dev/null | grep chr22 || echo "(none yet -- check task status above)"

## Stage 2 -- full run (all 22 chromosomes)

Run after Stage 1 succeeds. Same `--tasks` TSV pattern as `05`.

In [ ]:
CHR_LENGTHS = {
    1: 248_956_422, 2: 242_193_529, 3: 198_295_559, 4: 190_214_555,
    5: 181_538_259, 6: 170_805_979, 7: 159_345_973, 8: 145_138_636,
    9: 138_394_717, 10: 133_797_422, 11: 135_086_622, 12: 133_275_309,
    13: 114_364_328, 14: 107_043_718, 15: 101_991_189, 16: 90_338_345,
    17: 83_257_441, 18: 80_373_285, 19: 58_617_616, 20: 64_444_167,
    21: 46_709_983, 22: 50_818_468,
}
CHRS_LARGEST_FIRST = sorted(CHR_LENGTHS, key=CHR_LENGTHS.get, reverse=True)

TASKS_PATH = "/tmp/hm3_panel_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env CHR\t--env CHR_PGEN_GS\t--env CHR_PVAR_GS\t--env CHR_PSAM_GS\t--env OUT_NAME\n")
    for chr_num in CHRS_LARGEST_FIRST:
        out_name = f"chr{chr_num}_hm3_{CDR_VERSION}"
        f.write(
            f"{chr_num}\t{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pgen\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pvar\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.psam\t{out_name}\n"
        )

print(open(TASKS_PATH).read())

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$PLINK_BIN_GS" "$KG_HARMONIZE_GS" "$BUCKET_DIR_GS" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
PLINK_BIN_GS=$8
KG_HARMONIZE_GS=$9
BUCKET_DIR_GS=${10}
MACHINE_VCPUS=${11}
MEMORY_MB=${12}
ANCESTRY_BUCKET_DIR_GS=${13}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "hm3-panel-genome-wide" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --input KG_HARMONIZE="$KG_HARMONIZE_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --make-pgen \
      --out chr_relabeled

    grep -v "^##" chr_relabeled.pvar | awk "NR>1 {print \$3, \$4, \$5}" | LC_ALL=C sort > chr_id_ref_alt.sorted
    LC_ALL=C comm -12 chr_id_ref_alt.sorted "$KG_HARMONIZE" | awk "{print \$1}" > agreeing_snps.ids

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --extract agreeing_snps.ids \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  --tasks /tmp/hm3_panel_tasks.tsv \
  > /tmp/hm3_genome_wide_job_id.txt

cat /tmp/hm3_genome_wide_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/hm3_genome_wide_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Resubmit missing chromosomes only

Checks the bucket directly for what's missing, resubmits just those. Safe to rerun anytime.

In [ ]:
# gcsfuse-mounted local path to the same directory Batch's --output-recursive writes to
BUCKET_DIR = os.path.join(ANCESTRY_BUCKET_DIR, "ancestry_panel")

def chr_is_done(chr_num):
    prefix = os.path.join(BUCKET_DIR, f"chr{chr_num}_hm3_{CDR_VERSION}")
    return all(os.path.isfile(f"{prefix}.{ext}") for ext in ("pgen", "pvar", "psam"))

MISSING_CHRS = [c for c in range(1, 23) if not chr_is_done(c)]
print(f"{22 - len(MISSING_CHRS)}/22 chromosomes already persisted; missing: {MISSING_CHRS}")

RESUBMIT_TASKS_PATH = "/tmp/hm3_panel_tasks_resubmit.tsv"
with open(RESUBMIT_TASKS_PATH, "w") as f:
    f.write("--env CHR\t--env CHR_PGEN_GS\t--env CHR_PVAR_GS\t--env CHR_PSAM_GS\t--env OUT_NAME\n")
    for chr_num in MISSING_CHRS:
        out_name = f"chr{chr_num}_hm3_{CDR_VERSION}"
        f.write(
            f"{chr_num}\t{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pgen\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pvar\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.psam\t{out_name}\n"
        )

if MISSING_CHRS:
    print(open(RESUBMIT_TASKS_PATH).read())
else:
    print("Nothing to resubmit -- all 22 chromosomes already persisted. Skip the next cell and go to the merge section.")

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$PLINK_BIN_GS" "$KG_HARMONIZE_GS" "$BUCKET_DIR_GS" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
PLINK_BIN_GS=$8
KG_HARMONIZE_GS=$9
BUCKET_DIR_GS=${10}
MACHINE_VCPUS=${11}
MEMORY_MB=${12}
ANCESTRY_BUCKET_DIR_GS=${13}

if [ "$(tail -n +2 /tmp/hm3_panel_tasks_resubmit.tsv | wc -l)" -eq 0 ]; then
  echo "Nothing to resubmit -- all chromosomes already persisted."
  exit 0
fi

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "hm3-panel-resubmit" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --input KG_HARMONIZE="$KG_HARMONIZE_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --make-pgen \
      --out chr_relabeled

    grep -v "^##" chr_relabeled.pvar | awk "NR>1 {print \$3, \$4, \$5}" | LC_ALL=C sort > chr_id_ref_alt.sorted
    LC_ALL=C comm -12 chr_id_ref_alt.sorted "$KG_HARMONIZE" | awk "{print \$1}" > agreeing_snps.ids

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --extract agreeing_snps.ids \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  --tasks /tmp/hm3_panel_tasks_resubmit.tsv \
  > /tmp/hm3_resubmit_job_id.txt

cat /tmp/hm3_resubmit_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/hm3_resubmit_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Merge chromosomes into a genome-wide ancestry pfile

`--pmerge-list`, size-verified copy first (a truncated local copy broke `05`'s merge once). No bed export -- nothing downstream needs PLINK 1.9.

In [ ]:
import shutil

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_ancestry_panel")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

for chr_num in range(1, 23):
    for ext in ("pgen", "pvar", "psam"):
        bucket_path = os.path.join(BUCKET_DIR, f"chr{chr_num}_hm3_{CDR_VERSION}.{ext}")
        local_path = os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_hm3_{CDR_VERSION}.{ext}")
        assert os.path.isfile(bucket_path), f"missing persisted chromosome output: {bucket_path!r} -- rerun Stage 2/resubmit for chr{chr_num} first"
        needs_copy = not os.path.isfile(local_path) or os.path.getsize(local_path) != os.path.getsize(bucket_path)
        if needs_copy:
            shutil.copy(bucket_path, local_path)

print("All 22 chromosome trios present in local scratch (size-verified against the bucket).")

In [ ]:
MERGE_LIST_PATH = os.path.join(LOCAL_WORK_DIR, f"chr_merge_list_hm3_{CDR_VERSION}.txt")
with open(MERGE_LIST_PATH, "w") as f:
    for chr_num in range(1, 23):
        f.write(os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_hm3_{CDR_VERSION}") + "\n")

MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"ancestry_panel_hm3_{CDR_VERSION}")
PLINK2_LOCAL_BIN = os.path.expanduser("~/bin/plink2")
assert os.path.isfile(PLINK2_LOCAL_BIN), f"plink2 not found at {PLINK2_LOCAL_BIN!r} -- run the 'Stage the plink2 binary' cell above first"

print(MERGE_LIST_PATH)
print(MERGED_PREFIX)

In [ ]:
%%bash -s "$MERGE_LIST_PATH" "$MERGED_PREFIX" "$PLINK2_LOCAL_BIN"
set -e
MERGE_LIST_PATH=$1
MERGED_PREFIX=$2
PLINK2_LOCAL_BIN=$3

time "$PLINK2_LOCAL_BIN" \
  --pmerge-list "$MERGE_LIST_PATH" \
  --make-pgen \
  --out "$MERGED_PREFIX"

echo "Genome-wide HM3 variant count:"
grep -vc '^##' "${MERGED_PREFIX}.pvar"
echo "Sample count:"
wc -l < "${MERGED_PREFIX}.psam"

ls -lh "${MERGED_PREFIX}".*

In [ ]:
%%bash -s "$MERGED_PREFIX" "$BUCKET_DIR"
set -e
MERGED_PREFIX=$1
BUCKET_DIR=$2

mkdir -p "$BUCKET_DIR"
cp "${MERGED_PREFIX}".pgen "${MERGED_PREFIX}".pvar "${MERGED_PREFIX}".psam "${MERGED_PREFIX}".log "$BUCKET_DIR/"

ls -lh "$BUCKET_DIR"/"$(basename "$MERGED_PREFIX")".*

## Next steps

Run `03_round2_1000g_filter.ipynb`, then `04_final_pca.ipynb`. Runs once, ever -- not once per `BASE_GROUP`.